In [1]:
from src.autotune.tune import (
    tune_time_to_optimal
)
from src.autotune.models import SolverFactory, ModelSolver, SolutionStatus
from src.cpsat_autotune.cp_sat_solver import CpSatSolverFactory, CpSatSolver


In [2]:
# ImportError: attempted relative import beyond top-level package
import ortools
ortools.__version__

'9.14.6206'

In [3]:
from ortools.sat.python.cp_model import CpModel, CpSolver
model1 = CpModel()
model2 = CpModel()
models = [model1, model2]

In [4]:
class ArrayModelSolver(CpSatSolver):

    def solve(self, model : list[CpModel])->SolutionStatus:
        for m in model:
            self.solver.solve(m)
        return SolutionStatus.OPTIMAL

In [5]:
class ArrayFactory(CpSatSolverFactory):
    def prepare_solver(self, params: dict[str, float | int | bool | list | tuple]) -> ModelSolver[list[CpModel]]:
        return ArrayModelSolver(solver=self._prepare_cp_sat_solver(params))

In [6]:
from src.autotune.parameter_space import ParameterSpace
from src.autotune import tune
from src.cpsat_autotune.cp_sat_parameters import CPSAT_PARAMETER_SUGGESTIONS, CPSAT_PARAMETERS

In [7]:
# !pip uninstall ortools

In [8]:
parameter_space = ParameterSpace(
    all_parameters=CPSAT_PARAMETERS,
    parameters=CPSAT_PARAMETER_SUGGESTIONS
)
parameter_space.drop_parameter("use_lns_only")  # Not useful for this metric
parameter_space.drop_parameter("max_time_in_seconds")
parameter_space.filter_applicable_parameters(models)

2026-02-24 13:51:58,870 - INFO - Dropping parameter `search_branching` as it is not effective for any of the provided models.
2026-02-24 13:51:58,870 - INFO - Dropping parameter `optimize_with_core` as it is not effective for any of the provided models.


In [9]:
result = tune_time_to_optimal(
     model = models,
    solver_factory = ArrayFactory(),
    parameter_space = parameter_space,
    max_time_in_seconds = 10,
    relative_gap_limit= 0.0,
    n_samples_for_trial = 10,
    n_samples_for_verification = 30,
    n_trials= 100,
)
result

2026-02-24 13:51:58,873 - INFO - Starting tuning to minimize time to optimal solution.
2026-02-24 13:51:58,874 - INFO - Initialized Metric with direction: minimize
2026-02-24 13:51:58,874 - INFO - Starting hyperparameter tuning with 100 trials.
2026-02-24 13:51:58,874 - INFO - Evaluating with params: {}, num_runs: 30, knockout_score: None
2026-02-24 13:51:58,874 - INFO - Starting solver with random_seed: 1908344614, max_time_in_seconds: 10, relative_gap_limit: 0.0, absolute_gap_limit: 0.0
2026-02-24 13:51:58,880 - INFO - Solver completed in 0.005424 seconds with status: SolutionStatus.OPTIMAL
2026-02-24 13:51:58,880 - INFO - Starting solver with random_seed: 835367419, max_time_in_seconds: 10, relative_gap_limit: 0.0, absolute_gap_limit: 0.0
2026-02-24 13:51:58,883 - INFO - Solver completed in 0.003357 seconds with status: SolutionStatus.OPTIMAL
2026-02-24 13:51:58,884 - INFO - Starting solver with random_seed: 1646857135, max_time_in_seconds: 10, relative_gap_limit: 0.0, absolute_gap_

────────────────────────────────────────────── OPTIMIZED PARAMETERS ───────────────────────────────────────────────

┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ #   ┃ Parameter                     ┃ Value ┃ Contribution ┃ Default Value ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 1   │ use_phase_saving              │ False │    13.68%    │     True      │
│ 2   │ binary_minimization_algorithm │   3   │    13.62%    │       1       │
│ 3   │ minimization_algorithm        │   1   │    16.40%    │       2       │
│ 4   │ clause_cleanup_ordering       │   1   │    15.61%    │       0       │
│ 5   │ cp_model_probing_level        │   1   │    11.82%    │       2       │
│ 6   │ linearization_level           │   0   │    13.71%    │       1       │
│ 7   │ boolean_encoding_level        │   0   │    15.17%    │       1       │
└─────┴───────────────────────────────┴───────┴──────────────┴───────────────┘

────────────────────────────────────────────────── Descriptions ───────────────────────────────────────────────────

1. use_phase_saving When enabled, the solver remembers the last assigned value (phase) of each variable and tries  
to reuse it in future decisions. This can help the solver to quickly find solutions by reusing previously          
successful assignments.                                                                                            

2. binary_minimization_algorithm Specifies the algorithm used for binary clause minimization during conflict       
analysis. The options are:                                                                                         

 • 0 NO_BINARY_MINIMIZATION.                                                                                       
 • 1 BINARY_MINIMIZATION_FIRST                                                                                     
 • 2 BINARY_MINIMIZATION_WITH_REACHABILITY                                                                         
 • 3 EXPERIMENTAL_BINARY_MINIMIZATION                                                                              
 • 4 BINARY_MINIMIZATION_FIRST_WITH_TRANSITIVE_REDUCTION                                                           

3. minimization_algorithm Strategy for minimizing conflicts when creating them. The options are: - 0 NONE - 1      
SIMPLE - 2 RECURSIVE - 3 EXPERIMENTAL (deactivated because it can lead to segfaults)                               

4. clause_cleanup_ordering Specifies which clauses are kept during a cleanup. The options are: - 0 CLAUSE_ACTIVITY:
Keep orders by decreasing activity. - 1 CLAUSE_LBD: Keep orders by increasing LBD.                                 

5. cp_model_probing_level Defines the intensity of probing during presolve, where variables are temporarily fixed  
to infer more information about the problem. Higher levels of probing can result in a more simplified problem but  
require more computation time during presolve.                                                                     

6. linearization_level Controls the extent to which integer constraints are transformed into Boolean variables for 
LP relaxation. The levels are:                                                                                     

 • 0: No linearization, the solver does not use LP relaxation.                                                     
 • 1: Basic linearization, including linear constraints and fully encoded Boolean variables.                       
 • 2: More comprehensive linearization, including Boolean constraints.                                             

Increasing the linearization level can tighten the relaxation, but it also increases the complexity of the model.  

7. boolean_encoding_level Controls the eagerness to fully encode integer variables as Boolean variables. The higher
the level, the more aggressive the encoding.

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━━┓
┃ Metric                                    ┃ Mean ┃   Min ┃  Max ┃ #Samples ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━━━┩
│ Time in seconds with Default Parameters   │  0.0 │   0.0 │ 0.01 │       30 │
│ Time in seconds with Optimized Parameters │ -0.0 │ -0.09 │  0.0 │       30 │
└───────────────────────────────────────────┴──────┴───────┴──────┴──────────┘

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭──────────────────────────────────────────────────── WARNING ────────────────────────────────────────────────────╮
│                                                                                                                 │
│      The optimized parameters listed above were obtained based on a sampling approach                           │
│      and may not fully capture the complexities of the entire problem space.                                    │
│      While statistical reasoning has been applied, these results should be considered                           │
│      as a suggestion for further evaluation rather than definitive settings.                                    │
│                                                                                                                 │
│      It is strongly recommended to validate these parameters in larger, more comprehensive                      │
│      experiments before adopting them in critical applications.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-02-24 13:52:02,083 - INFO - Hyperparameter tuning completed.
2026-02-24 13:52:02,083 - INFO - Tuning for time to optimal completed.


{'use_phase_saving': False,
 'binary_minimization_algorithm': 3,
 'minimization_algorithm': 1,
 'clause_cleanup_ordering': 1,
 'cp_model_probing_level': 1,
 'linearization_level': 0,
 'boolean_encoding_level': 0}